In [ ]:
# 1. Clone repo của nhóm về Colab
!git clone https://github.com/dangthihoangnganit6/V-ChartText.git

# 2. Chuyển vào thư mục repo
%cd V-ChartText

# 3. Cài đặt các thư viện cần thiết
!pip install -q transformers accelerate torchvision qwen-vl-utils gradio

Cloning into 'V-ChartText'...
remote: Enumerating objects: 185, done.
remote: Total 185 (delta 0), reused 0 (delta 0), pack-reused 185 (from 1)
Receiving objects: 100% (185/185), 83.03 MiB | 33.71 MiB/s, done.
Resolving deltas: 100% (34/34), done.
/content/V-ChartText/V-ChartText


In [ ]:
!ls -la /content/V-ChartText

total 128
drwxr-xr-x 10 root root  4096 Nov  8 04:42 .
drwxr-xr-x  1 root root  4096 Nov  8 03:59 ..
drwxr-xr-x  4 root root  4096 Nov  8 03:43 cookbooks
drwxr-xr-x  2 root root  4096 Nov  8 03:43 docker
drwxr-xr-x  7 root root  4096 Nov  8 03:43 evaluation
drwxr-xr-x  8 root root  4096 Nov  8 03:43 .git
drwxr-xr-x  2 root root  4096 Nov  8 03:48 .gradio
-rw-r--r--  1 root root 11357 Nov  8 03:43 LICENSE
drwxr-xr-x  6 root root  4096 Nov  8 03:43 qwen-vl-finetune
drwxr-xr-x  3 root root  4096 Nov  8 03:43 qwen-vl-utils
-rw-r--r--  1 root root 52828 Nov  8 03:43 README.md
-rw-r--r--  1 root root   305 Nov  8 03:43 requirements_web_demo.txt
drwxr-xr-x  8 root root  4096 Nov  8 04:42 V-ChartText
-rw-r--r--  1 root root 14547 Nov  8 03:43 web_demo_mm.py


In [ ]:
import gradio as gr
import json
import re
from PIL import Image
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from qwen_vl_utils import process_vision_info

# Trỏ đến checkpoint của nhóm (ví dụ: "Qwen/Qwen2-VL-2B-Instruct" hoặc checkpoint Qwen3-VL bạn tải về)
MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

print("Đang nạp processor và model...")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
print("Mô hình đã sẵn sàng!")

DEFAULT_PROMPT = """Trích xuất dữ liệu biểu đồ thành JSON hợp lệ theo cấu trúc:
{
  "chart_title": "tiêu đề",
  "chart_type": "column",
  "x_axis_label": "trục hoành",
  "categories": ["mục 1", "mục 2"],
  "unit": "đơn vị",
  "series": [
    {
      "name": "chuỗi dữ liệu",
      "values": [0, 0]
    }
  ]
}
Quy tắc:
- values là mảng số (float/int).
- Dấu thập phân dùng '.', bỏ dấu chấm/phẩy phân cách hàng nghìn.
- Chỉ trả về chuỗi JSON, không giải thích gì thêm."""

def extract_chart(image, custom_prompt):
    if image is None:
        return "Vui lòng tải ảnh biểu đồ lên!", "{}"

    prompt = custom_prompt if (custom_prompt and custom_prompt.strip()) else DEFAULT_PROMPT

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt}
            ]
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=512)

    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    raw_output = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

    clean_text = re.sub(r"^```json\s*|\s*```$", "", raw_output.strip(), flags=re.M)
    try:
        parsed = json.loads(clean_text)
        formatted_json = json.dumps(parsed, indent=2, ensure_ascii=False)
        status = "✅ Parse JSON thành công!"
    except Exception as e:
        formatted_json = clean_text
        status = f"⚠️ Lỗi cú pháp JSON ({e}):"

    return status, formatted_json

with gr.Blocks(title="V-ChartText Demo") as demo:
    gr.Markdown("# 📊 V-ChartText Demo")
    with gr.Row():
        with gr.Column():
            input_image = gr.Image(type="pil", label="Ảnh biểu đồ")
            input_prompt = gr.Textbox(lines=6, label="Prompt cấu trúc", value=DEFAULT_PROMPT)
            btn = gr.Button("Trích xuất dữ liệu", variant="primary")
        with gr.Column():
            output_status = gr.Textbox(label="Trạng thái kiểm tra")
            output_json = gr.Code(label="Kết quả JSON", language="json")

    btn.click(fn=extract_chart, inputs=[input_image, input_prompt], outputs=[output_status, output_json])

demo.launch(share=True, debug=True)

Đang nạp processor và model...


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Mô hình đã sẵn sàng!
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ed3263f77288797864.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://ed3263f77288797864.gradio.live


In [ ]:
from google.colab import drive, auth
from google.auth import default
import gspread

# 1. Mount Drive để lấy ảnh
drive.mount('/content/drive')

# 2. Cài thư viện & xác thực tài khoản Google để ghi vào Sheet
!pip install -q gspread
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
print("Đã kết nối Google Drive và Google Sheets thành công!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Đã kết nối Google Drive và Google Sheets thành công!


In [ ]:
!ls "/content/drive/MyDrive/Ảnh biểu đồ"

'Biểu đồ cột_74 ảnh_đủ r'      'Biểu đồ miền_24 ảnh_đủ r'
'Biểu đồ đường_52 ảnh_đủ'      'Biểu đồ tròn_42 ảnh_đủ'
'Biểu đồ kết hợp_21 ảnh_đủ r'   Checklist.gsheet


In [ ]:
import os
import re
import json
from PIL import Image
from google.colab import drive, auth
from google.auth import default
import gspread

# ==========================================
# 1. MOUNT DRIVE & CẤP QUYỀN ĐẦY ĐỦ CHO SHEETS
# ==========================================
print("1. Đang kết nối Google Drive & cấp quyền Sheets...")
drive.mount('/content/drive')

auth.authenticate_user()
# Bắt buộc khai báo scopes để tránh triệt để lỗi 404 Entity Not Found
creds, _ = default(scopes=[
    'https://www.googleapis.com/auth/spreadsheets',
    'https://www.googleapis.com/auth/drive'
])
gc = gspread.authorize(creds)

# ==========================================
# 2. MỞ FILE CHECKLIST & TAB DATA_LABELING
# ==========================================
print("2. Đang mở Google Sheet...")
SHEET_ID = "1m9Ctjh-YiMqOIdUobRSmeShj5Tx445JKVHuxPEnnOU4"
wb = gc.open_by_key(SHEET_ID)
sheet = wb.worksheet("Data_Labeling")
print(f"-> Kết nối thành công tới Sheet: '{wb.title}' | Tab: '{sheet.title}'")

# ==========================================
# 3. QUÉT TOÀN BỘ FILE ẢNH TRONG THƯ MỤC TỔNG
# ==========================================
DRIVE_ROOT = "/content/drive/MyDrive/Ảnh biểu đồ/Biểu đồ miền_24 ảnh_đủ r"
print(f"3. Đang quét ảnh trong: {DRIVE_ROOT}...")

import unicodedata
def norm(s):
    return unicodedata.normalize("NFC", s).strip().lower()

image_path_map = {}
for root, _, files in os.walk(DRIVE_ROOT):
    for f in files:
        if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
            image_path_map[norm(f)] = os.path.join(root, f)

print(f"-> Đã lập chỉ mục {len(image_path_map)} file ảnh sẵn sàng trích xuất.")

# ==========================================
# 4. HÀM GỌI MODEL & PARSE JSON
# ==========================================
PROMPT_JSON = """Bạn là trợ lý trích xuất số liệu biểu đồ học tập tiếng Việt.
Biểu đồ giáo dục Việt Nam có quy ước dấu chấm '.' và phẩy ',' không đồng nhất. Vì vậy, BẮT BUỘC phải đọc các mốc số trên trục tung (Y-axis) để làm mốc đối chiếu trước khi đọc giá trị.

Hãy phân tích hình ảnh và trả về DUY NHẤT một chuỗi JSON theo đúng 7 trường:
{
  "chart_title": "Tiêu đề biểu đồ (nếu không có ghi 'Biểu đồ không có tên')",
  "chart_type": "column | cột ghép | cột chồng | đường | miền | tròn | kết hợp",
  "unit": "Đơn vị đo lường (vd: Nghìn tấn, triệu USD, %, mm, không có)",
  "x_axis_label": "Nhãn trục hoành (vd: Năm, Tỉnh/Thành phố, không có)",
  "categories": "Danh sách các mốc/đối tượng trục hoành, phân tách bằng ' | '",
  "series_name": "Tên các chuỗi dữ liệu trong chú giải, phân tách bằng ' | '",
  "values": "Dãy giá trị đọc được trên biểu đồ. Phân tách các số trong cùng 1 chuỗi bằng ' | ', giữa các chuỗi bằng ' || '"
}

QUY TẮC BẮT BUỘC ĐỂ KHÔNG ĐOÁN BỪA SỐ LIỆU:
1. Đọc và đối chiếu với trục tung (Y-axis):
   - Nhìn thang chia vạch trên trục tung (ví dụ các mốc: 0, 500, 1000, 1500 hay 0, 1, 2, 3, 4, 5).
   - Dựa vào độ cao của cột/đường so với thang trục tung để xác định giá trị thực tế.
2. Cách ghi trường 'values':
   - GIỮ NGUYÊN chuỗi ký tự số nhìn thấy trên hình, KHÔNG tự ý xóa dấu hay đổi dấu (nếu hình ghi '1.200' hãy ghi '1.200', ghi '12,5' hãy ghi '12,5', ghi '6 586,6' hãy ghi '6 586,6').
   - Tuyệt đối không tự suy diễn đổi '1.200' thành '1.2' nếu cột đó đang cao vượt mốc 1000 trên trục tung.
3. Nếu là biểu đồ tròn không có trục tung:
   - Tổng cơ cấu thường là 100%. Nếu các số có dạng '45,2' và '54,8' thì dấu phẩy chắc chắn là số thập phân.
4. Giữ nguyên tiếng Việt: Không dịch chú giải hoặc tên đối tượng sang tiếng Anh.
5. Chỉ trả về một khối JSON thuần duy nhất.
"""

def extract_chart_data(img_path):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img_path},
                {"type": "text", "text": PROMPT_JSON}
            ]
        }
    ]
    text_input = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text_input],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to("cuda")

    generated_ids = model.generate(**inputs, max_new_tokens=512)
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    raw_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

    # Bóc tách JSON an toàn
    match = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", raw_text)
    clean_str = match.group(1) if match else raw_text.strip()
    return json.loads(clean_str)

# ==========================================
# 5. DUYỆT TỪNG DÒNG SHEET & GHI KẾT QUẢ TỰ ĐỘNG
# ==========================================
rows = sheet.get_all_values()
print(f"4. Bắt đầu xử lý {len(rows) - 3} dòng dữ liệu trên Sheet...")

for idx in range(3, len(rows)):
    row_num = idx + 1
    row = rows[idx]

    img_name = row[1].strip()
    key = norm(img_name)
    ...
    if key not in image_path_map:
        continue   # bỏ qua các ảnh không nằm trong thư mục đang chạy
    img_path = image_path_map[key]

    try:
        data = extract_chart_data(img_path)

        c_title = data.get("chart_title", "")
        unit = data.get("unit", "")
        x_label = data.get("x_axis_label", "")
        categories = " | ".join([str(c) for c in data.get("categories", [])])
        series_name = data.get("series_name", "")
        values = " | ".join([str(v) for v in data.get("values", [])])

        # Cập nhật các cột C đến K (từ chart_title đến Trạng thái)
        update_vals = [
            c_title,
            row[3],          # giữ nguyên chart_type
            unit,
            x_label,
            categories,
            series_name,
            values,
            "AI_Auto",       # Người gán
            "Cần kiểm tra"   # Trạng thái
        ]
        sheet.update(f"C{row_num}:K{row_num}", [update_vals])
        print(f"✅ Dòng {row_num} [{img_name}]: Xong")

    except Exception as e:
        sheet.update_cell(row_num, 11, "Lỗi đọc AI")
        print(f"❌ Dòng {row_num} [{img_name}]: Lỗi - {str(e)}")

print("\n🎉 HOÀN TẤT! Toàn bộ bảng đã được AI điền dữ liệu tự động.")

1. Đang kết nối Google Drive & cấp quyền Sheets...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
2. Đang mở Google Sheet...
-> Kết nối thành công tới Sheet: 'Checklist' | Tab: 'Data_Labeling'
3. Đang quét ảnh trong: /content/drive/MyDrive/Ảnh biểu đồ/Biểu đồ miền_24 ảnh_đủ r...
-> Đã lập chỉ mục 24 file ảnh sẵn sàng trích xuất.
4. Bắt đầu xử lý 214 dòng dữ liệu trên Sheet...


/tmp/ipykernel_1070/2690875229.py:140: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  sheet.update(f"C{row_num}:K{row_num}", [update_vals])


✅ Dòng 120 [Biểu đồ miền_1.webp]: Xong
✅ Dòng 121 [Biểu đồ miền_2.jpg]: Xong
✅ Dòng 122 [Biểu đồ miền_3.jpg]: Xong
✅ Dòng 123 [Biểu đồ miền_4.jpg]: Xong
✅ Dòng 124 [Biểu đồ miền_5.png]: Xong
✅ Dòng 125 [Biểu đồ miền_6.png]: Xong
✅ Dòng 126 [Biểu đồ miền_7.jpg]: Xong
✅ Dòng 127 [Biểu đồ miền_8.jpg]: Xong
✅ Dòng 128 [Biểu đồ miền_9.webp]: Xong
✅ Dòng 129 [Biểu đồ miền_10.png]: Xong
✅ Dòng 130 [Biểu đồ miền_11.png]: Xong
✅ Dòng 131 [Biểu đồ miền_12.png]: Xong
✅ Dòng 132 [Biểu đồ miền_13.png]: Xong
✅ Dòng 133 [Biểu đồ miền_14.png]: Xong
✅ Dòng 134 [Biểu đồ miền_15.png]: Xong
✅ Dòng 135 [Biểu đồ miền_16.png]: Xong
✅ Dòng 136 [Biểu đồ miền_17.jpg]: Xong
✅ Dòng 137 [Biểu đồ miền_18.jpg]: Xong
✅ Dòng 138 [Biểu đồ miền_19.png]: Xong
✅ Dòng 139 [Biểu đồ miền_20.png]: Xong
✅ Dòng 140 [Biểu đồ miền_21.png]: Xong
✅ Dòng 141 [Biểu đồ miền_22.jpeg]: Xong
✅ Dòng 142 [Biểu đồ miền_23.jpg]: Xong
✅ Dòng 143 [Biểu đồ miền_24.png]: Xong

🎉 HOÀN TẤT! Toàn bộ bảng đã được AI điền dữ liệu tự động.


In [ ]:
import os
import re
import json
import csv
import pandas as pd
from PIL import Image

# 1. QUÉT FILE ẢNH TRÊN DRIVE
DRIVE_ROOT = "/content/drive/MyDrive/Ảnh biểu đồ"
print(f"1. Đang quét ảnh trong: {DRIVE_ROOT}...")

image_path_map = {}
for root, _, files in os.walk(DRIVE_ROOT):
    for f in files:
        if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
            image_path_map[f.strip()] = os.path.join(root, f)

print(f"-> Đã tìm thấy {len(image_path_map)} file ảnh.")

# 2. PROMPT TRÍCH XUẤT CHO VLM
PROMPT_JSON = """Bạn là trợ lý trích xuất dữ liệu biểu đồ. Hãy phân tích hình ảnh và trả về DUY NHẤT một chuỗi JSON hợp lệ theo đúng cấu trúc sau (không kèm lời giải thích hay markdown):
{
  "chart_title": "Tên đầy đủ của biểu đồ",
  "unit": "Đơn vị đo",
  "x_axis_label": "Tên trục hoành hoặc tên đối tượng",
  "categories": ["Mục 1", "Mục 2"],
  "series_name": "Tên chuỗi dữ liệu",
  "values": [123.4, 567.8]
}
Quy tắc:
- Số thập phân dùng dấu chấm '.'
- Bỏ toàn bộ dấu cách và dấu phân cách hàng nghìn.
- Chỉ trả về chuỗi JSON thuần.
"""

def extract_chart_data(img_path):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img_path},
                {"type": "text", "text": PROMPT_JSON}
            ]
        }
    ]
    text_input = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text_input],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to("cuda")

    generated_ids = model.generate(**inputs, max_new_tokens=512)
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    raw_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

    match = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", raw_text)
    clean_str = match.group(1) if match else raw_text.strip()
    return json.loads(clean_str)

# 3. CHẠY TRÍCH XUẤT TOÀN BỘ DANH SÁCH ẢNH
results = []
# Sắp xếp tên file tự nhiên
sorted_img_names = sorted(list(image_path_map.keys()))

print(f"2. Bắt đầu xử lý {len(sorted_img_names)} ảnh...")

for idx, img_name in enumerate(sorted_img_names, start=1):
    img_path = image_path_map[img_name]

    # Nhận diện sơ bộ chart_type từ tên file
    lower_name = img_name.lower()
    chart_type = "column"
    if "đường" in lower_name or "line" in lower_name:
        chart_type = "line"
    elif "tròn" in lower_name or "pie" in lower_name:
        chart_type = "pie"
    elif "miền" in lower_name or "area" in lower_name:
        chart_type = "area"

    try:
        data = extract_chart_data(img_path)
        c_title = data.get("chart_title", "")
        unit = data.get("unit", "")
        x_label = data.get("x_axis_label", "")
        categories = " | ".join([str(c) for c in data.get("categories", [])])
        series_name = data.get("series_name", "")
        values = " | ".join([str(v) for v in data.get("values", [])])
        status = "Cần kiểm tra"
        print(f"✅ [{idx}/{len(sorted_img_names)}] Xong: {img_name}")
    except Exception as e:
        c_title, unit, x_label, categories, series_name, values = "", "", "", "", "", ""
        status = "Lỗi đọc AI"
        print(f"❌ [{idx}/{len(sorted_img_names)}] Lỗi: {img_name} ({e})")

    results.append({
        "STT": idx,
        "image_name": img_name,
        "chart_title": c_title,
        "chart_type": chart_type,
        "unit": unit,
        "x_axis_label": x_label,
        "categories": categories,
        "series_name": series_name,
        "values": values,
        "Người gán": "AI_Auto",
        "Trạng thái kiểm tra": status
    })

# 4. XUẤT FILE VÀ TỰ ĐỘNG TẢI VỀ
df = pd.DataFrame(results)
output_path = "/content/data_labeling_ai.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"\n🎉 Đã xuất thành công file: {output_path}")

from google.colab import files
files.download(output_path)